## Libraries and Preparation

In [27]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"

import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.transforms.functional as F_transforms
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2
import graphviz
import pytorch_lightning as pl
import torchmetrics
import wandb
from tqdm import tqdm
from PIL import Image
from torchvision import models

!pip install torchview
from torchview import draw_graph
from scipy.ndimage import gaussian_laplace
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from torchvision.transforms import ToTensor

from torch.utils.data import Dataset
from torchvision import datasets
import timm
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
import imagehash

%matplotlib inline

In [28]:
DEVICE = "cuda"

In [29]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('cats-and-dogs')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/cats-and-dogs


In [30]:
train_df = pd.read_csv('/kaggle/input/competitions/cats-and-dogs/train.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/competitions/cats-and-dogs/train.csv'

## Exploratory Data Analysis

In [ ]:
label_cols = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
train_df['num_labels'] = train_df[label_cols].sum(axis=1)
#train_df[(train_df['Moris'] == 1) & (train_df['Motya'] == 1)]
train_df.head(5)

In [ ]:
image = cv2.imread('/kaggle/input/competitions/cats-and-dogs/train/7e17a6e589f9c891857b32bbb3652d19b7d5a215.jpg')

plt.imshow(image)

**QUESTION: "Do we have duplicates by image IDs?"**

In [ ]:
print(f"Overall count: {train_df['image_id'].count()}")
print(f"Distinct count: {train_df['image_id'].nunique()}")

The training dataset contains 12,260 total samples with 12,260 distinct image_id entries, confirming there are no duplicate records or primary key collisions.

**QUESTION: "How many visual duplicates do we have?"**

In [ ]:
import imagehash

hash_dict = {}
duplicate_groups = []

for img_id in tqdm(train_df['image_id'].values):
    img_path = f'/kaggle/input/competitions/cats-and-dogs/train/{img_id}.jpg'
    
    with Image.open(img_path) as img:
        img_hash = str(imagehash.phash(img))
        
    if img_hash in hash_dict:
        hash_dict[img_hash].append(img_id)
    else:
        hash_dict[img_hash] = [img_id]

for h, id_list in hash_dict.items():
    if len(id_list) > 1:
        duplicate_groups.append(id_list)

print(f"\Found {len(duplicate_groups)} duplicates.")

if duplicate_groups:
    for i, group in enumerate(duplicate_groups[:5]):
        print(f"Group {i+1}: {group}")

Here we have a clear indication of possible data leakage. We have 124 visual duplicates and if we will not erase them during preprocessing, we can get the same image in train and validation sets. The model will "memorize" the images, which will artificially inflate our local metric (CV) but lead to failure on the actual Kaggle test set.

**QUESTION: "How many labels per row?"**

In [ ]:
labels_per_image_count = train_df['num_labels'].value_counts().sort_index()
labels_per_image_prop = train_df['num_labels'].value_counts(normalize=True).sort_index() * 100

dist_df = pd.DataFrame({
    'Count': labels_per_image_count,
    'Percentage (%)': labels_per_image_prop.round(2)
})

dist_df

The label distribution reveals that despite the multi-label setup, the dataset behaves predominantly as a single-label problem, with 81.35% of images containing exactly one label. Additionally, a significant 17.63% of the data contains zero target labels, acting as a background class that requires the model to be well-calibrated against false positives. True multi-label instances containing two or more labels are extremely sparse at roughly 1% of the dataset, which strictly necessitates multi-label stratified cross-validation to ensure these rare combinations are properly represented across all training and validation splits.

**QUESTION: "How many labels do we have per category?**

In [ ]:
temp_df = train_df[label_cols].copy()
temp_df['Other'] = (temp_df.sum(axis=1) == 0).astype(int)

all_cols = label_cols + ['Other']

categories_summary = pd.DataFrame({
    'Count': temp_df[all_cols].sum(),
    'Percentage (%)': (temp_df[all_cols].mean() * 100).round(2)
})

categories_summary.loc['Total'] = categories_summary.sum()
categories_summary

This distribution represents inbalanced data. Standard random splitting will likely result in validation folds that miss some of these rare classes entirely. Therefore, it is important to apply stratified CV in this case. Moreover this severe class skew necessitates specialized multi-label handling, such as loss weighting.

**QUESTION: "What shapes do we have across the images?"**

In [ ]:
pil_shapes = []
for img_name in tqdm(train_df['image_id'].values):
    img_path = os.path.join("/kaggle/input/competitions/cats-and-dogs/train", f"{img_name}.jpg")
    with Image.open(img_path) as img:
        pil_shapes.append(img.size) 

print(pd.Series(pil_shapes).value_counts())

Analysis of image dimensions across the entire dataset revealed a uniform spatial resolution of $375 \times 450$ pixels (Height $\times$ Width) for all 12,260 training samples. Consequently, no custom padding or dynamic resizing per batch is required. However, we should convert it to square size when training.

**QUESTION: "How does every label look?"**

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, label in enumerate(label_cols):
    img_id = train_df[train_df[label] == 1]['image_id'].iloc[0]
    img_path = f'/kaggle/input/competitions/cats-and-dogs/train/{img_id}.jpg'
    
    image = cv2.imread(img_path)
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(image)
    axes[i].set_title(label, fontsize=14, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

Two cats on both labels. Let's look where is Moris and where is Motya

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
used_image_ids = set()

for i, label in enumerate(label_cols):
    condition = ((train_df[label] == 1) & (~train_df['image_id'].isin(used_image_ids)))
    
    filtered_df = train_df[condition]
    
    if len(filtered_df) > 0:
        img_id = filtered_df['image_id'].iloc[0]
    else:
        fallback_df = train_df[(train_df[label] == 1) & (~train_df['image_id'].isin(used_image_ids))]
        if len(fallback_df) > 0:
            img_id = fallback_df['image_id'].iloc[0]
        else:
            img_id = train_df[train_df[label] == 1]['image_id'].iloc[0]
            
    used_image_ids.add(img_id)
    
    img_path = f'/kaggle/input/competitions/cats-and-dogs/train/{img_id}.jpg'
    
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(image)
    axes[i].set_title(label, fontsize=14, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

**QUESTION: "Do we have same labels on duplicate images?"**

In [ ]:
conflict_groups = []

for group in duplicate_groups:
    group_df = train_df[train_df['image_id'].isin(group)]
    
    unique_labels = group_df[label_cols].drop_duplicates()
    
    if len(unique_labels) > 1:
        conflict_groups.append(group_df)

print(f"Found conflict groups: {len(conflict_groups)}")

if len(conflict_groups) > 0:
    for i, df_conf in enumerate(conflict_groups[:3]):
        print(f"\n--- Conflict group {i+1} ---")
        print(df_conf[['image_id'] + label_cols])
else:
    print("All visual duplicates have identical labels")

Therefore, we can randomly delete on of two images in a duplicate group

## Metric Justification

**Proposed Metric: F1 Score**

The core strengh of proposed macro F1 metric is that it calculates the F1 score for each of the 5 classes independently and then takes the unweighted average. This forces the model to treat the rare classes as equally important as the dominant classes. It actively punishes models that lazily predict only cats and dogs.

However, a single misclassification or a noisy label in a tiny class can cause its individual F1-score to plummet. Since we have less than 1% of rare class, this moise is very possible to happen.

**Complementary Metrics**

It is worth computing Micro F1 along with Macro F1 to measure overall global accuracy without the heavy penalty of the rare classes.

Additionally, tracking Binary Cross-Entropy Loss is crucial because it evaluates continuous probabilities rather than hard thresholds, showing if the model is actually learning even if the F1 score hasn't improved yet.

**Alternative metrics and Justification**

As an alternative metric PR-AUC acts as an ideal alternative metric for this dataset because it effectively handles extreme class imbalance by completely ignoring True Negatives, which are overwhelmingly abundant in this multi-label task. Unlike ROC-AUC, which can show a misleadingly high score due to the massive number of easy negative predictions, PR-AUC focuses exclusively on the model's accuracy in predicting true positives in all predicted positives and its ability to find all actual positives across all probability thresholds. This ensures an objective evaluation of rare classes without being artificially affected by the dominant background class, while also avoiding the chaotic fluctuations caused by the hard thresholding of the Macro F1 score.

## Validation



**Validation Strategy** 

Multilabel Stratified 5-Fold Cross-Validation

**Why this?** 

The dataset demonstrates an extreme imbalance with rare classes appearing in less than 1% of instances and a multi-label structure. Simple random splitting or standard stratification could result in a rare class being entirely absent from a validation fold. Iterative multi-label stratification ensures that the proportions of both individual labels and their rare intersections are preserved across all folds.

**Correlation with leaderboard results**

We expect that multi-label stratification successfully balances the complexity of the folds, and strong augmentations prevent overfitting to the dominant classes Cat and Dog.

However, since the macro F1 score averages results across classes without accounting for their size, we can end up with a significantly lower score due to just a few errors on rare classes.

## Adversarial Validation (optional)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
backbone.classifier = torch.nn.Identity()
backbone.to(device)
backbone.eval()

transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_subset = train_df.sample(n=500, random_state=42).copy()
train_subset['is_test'] = 0

test_df = pd.read_csv("/kaggle/input/competitions/cats-and-dogs/test.csv")
test_subset = test_df.sample(n=500, random_state=42).copy()
test_subset['is_test'] = 1

adv_df = pd.concat([train_subset, test_subset]).reset_index(drop=True)

features = []
labels = []

for idx, row in tqdm(adv_df.iterrows(), total=len(adv_df)):
    img_id = row['image_id']
    is_test = row['is_test']
    
    if is_test == 0:
        img_path = f'/kaggle/input/competitions/cats-and-dogs/train/{img_id}.jpg'
    else:
        img_path = f'/kaggle/input/competitions/cats-and-dogs/test/{img_id}.jpg'
        
    try:
        with Image.open(img_path).convert('RGB') as img:
            tensor_img = transform(img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                feat = backbone(tensor_img).cpu().numpy().flatten()
                
            features.append(feat)
            labels.append(is_test)
    except Exception as e:
        continue

features = np.array(features)
labels = np.array(labels)

X_train, X_val, y_train, y_val = train_test_split(
    features, 
    labels, 
    test_size=0.3, 
    random_state=42, 
    stratify=labels
)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

preds = clf.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_val, preds)

print(f"\nReal Adversarial Validation ROC-AUC Score: {auc_score:.4f}")

Finally, there is no domain shift in train and test data

## Preprocessing

In [ ]:
# 1. Duplicate Removal

ids_to_drop = []
for group in duplicate_groups:
    ids_to_drop.extend(group[1:])

train_df = train_df[~train_df['image_id'].isin(ids_to_drop)].reset_index(drop=True)



# 2. Two-layer label encoder

class HierarchicalLabelEncoder:
    def __init__(self, label_columns):
        self.label_columns = label_columns
        
    def encode(self, row):
        labels = row[self.label_columns].values.astype(np.float32)

        if labels.sum() == 0:
            return labels
        if labels[2] == 1 or labels[3] == 1:  # Moris or Motya
            labels[0] = 1  # Cat
        if labels[4] == 1:                     # Biatrix
            labels[1] = 1  # Dog
            
        return labels

encoder = HierarchicalLabelEncoder(label_cols)



# 3. Square padding + resize

class SquarePad:
    def __call__(self, image):
        w, h = image.size
        max_wh = max(w, h)
        hp = (max_wh - w) // 2
        vp = (max_wh - h) // 2
        padding = (hp, vp, max_wh - w - hp, max_wh - h - vp)
        return F_transforms.pad(image, padding, fill=0, padding_mode='constant')


# Final transformation pipeline
transform = transforms.Compose([
    SquarePad(),
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## CNN

In [ ]:
CLASSES = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']

Model options: efficientnet_b3, convnext_tiny, resnet50, convnextv2_tiny, regnety_016, resnet50d

In [ ]:
MODEL_NAME = "convnext_tiny"
EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 0.05
BATCH_SIZE = 32
DROP_PATH = 0.1
FOLD = 0
DATA_PATH = '/kaggle/input/competitions/cats-and-dogs/train'

In [ ]:
def create_folds(input_df):
    mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=95431)
    input_df["fold"] = -1
    for fold, (_, val_idx) in enumerate(mskf.split(input_df, input_df[CLASSES].values)):
        input_df.loc[val_idx, "fold"] = fold
    return input_df

class CatDogDataset(Dataset):
    def __init__(self, input_df, label_cols, path, transform=None):
        self.input_df = input_df
        self.label_cols = label_cols
        self.path = path
        self.transform = transform

        if transform is None:
            raise ValueError("transform is a necessity")

    def __len__(self):
        return len(self.input_df)

    def __getitem__(self, idx):
        row = self.input_df.iloc[idx]
        
        img_id = row['image_id']
        img_path = os.path.join(self.path, f"{img_id}.jpg")
        image = Image.open(img_path).convert('RGB')

        img_tensor = self.transform(image)
        
        labels = row[self.label_cols].values.astype('float32')
        labels_tensor = torch.tensor(labels, dtype=torch.float32)

        return img_tensor, labels_tensor

In [ ]:
train_df = create_folds(train_df)
train_df[label_cols] = np.stack(train_df.apply(encoder.encode, axis=1).values)

train_part = train_df[train_df["fold"] != FOLD].reset_index(drop=True)
val_part = train_df[train_df["fold"] == FOLD].reset_index(drop=True)
print(len(train_part), len(val_part))

train_dataset = CatDogDataset(train_part, CLASSES, DATA_PATH, transform=transform)
val_dataset = CatDogDataset(val_part, CLASSES, DATA_PATH, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False,
                        num_workers=2, pin_memory=True)

pos = train_part[CLASSES].sum().values.astype(np.float32)
pos_weight = torch.tensor((len(train_part) - pos) / np.clip(pos, 1, None),
                          dtype=torch.float32, device=DEVICE)

print("pos_weight:", pos_weight.cpu().numpy().round(2))

In [ ]:
def build_model(model_name, drop_path=DROP_PATH):
    try:
        return timm.create_model(model_name, pretrained=True,
                                 num_classes=len(CLASSES),
                                 drop_path_rate=drop_path)
    except TypeError:
        return timm.create_model(model_name, pretrained=True,
                                 num_classes=len(CLASSES))

def build_training(model_name, lr=LR, epochs=EPOCHS):
    model = build_model(model_name).to(DEVICE)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=epochs * len(train_loader), pct_start=0.1)
    
    scaler = torch.amp.GradScaler("cuda")
    return model, criterion, optimizer, scheduler, scaler

In [ ]:
#parent probability cannot be smaller than probability of an offspring
def enforce_hierarchy(p):
    p = p.copy()
    p[:, 0] = np.maximum(p[:, 0], np.maximum(p[:, 2], p[:, 3]))
    p[:, 1] = np.maximum(p[:, 1], p[:, 4])
    return p

def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    model.train()
    running_loss = 0.0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            loss = criterion(model(X), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * X.size(0)
    return running_loss / len(loader.dataset)

In [ ]:
@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    logits_all, targets_all = [], []
    for X, y in loader:
        X = X.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(X)
        logits_all.append(logits.float().cpu())
        targets_all.append(y)

    logits = torch.cat(logits_all)
    targets = torch.cat(targets_all)

    bce = F.binary_cross_entropy_with_logits(logits, targets).item()
    probs = enforce_hierarchy(torch.sigmoid(logits).numpy())
    t = targets.numpy().astype(int)
    preds = (probs > 0.5).astype(int)

    macro = f1_score(t, preds, average="macro", zero_division=0)
    micro = f1_score(t, preds, average="micro", zero_division=0)
    return macro, micro, bce, probs, t

In [ ]:
model, criterion, optimizer, scheduler, scaler = build_training(MODEL_NAME)

best_macro, best_state = -1.0, None
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer,
                             scheduler, scaler, DEVICE)
    macro, micro, val_bce, _, _ = validate(model, val_loader, DEVICE)
    print(f"epoch {epoch+1:02d} | train_loss {train_loss:.4f} | val_bce {val_bce:.4f} "
          f"| val_macro_f1 {macro:.4f} | val_micro_f1 {micro:.4f}")

    if macro > best_macro:
        best_macro = macro
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, f"{MODEL_NAME}_fold{FOLD}_best.pt")

model.load_state_dict(best_state)
print(MODEL_NAME, "best val macro F1:", round(best_macro, 4))

In [ ]:
_, _, _, val_probs, val_targets = validate(model, val_loader, DEVICE)

grid = np.arange(0.05, 0.96, 0.01)
best_thresholds = np.array([
    grid[int(np.argmax([f1_score(val_targets[:, c], val_probs[:, c] >= t, zero_division=0)
                        for t in grid]))]
    for c in range(len(CLASSES))
])
print("Thresholds:", dict(zip(CLASSES, best_thresholds.round(2))))
print("Macro F1 @0.5:  ", f1_score(val_targets, (val_probs >= 0.5).astype(int),
                                   average="macro", zero_division=0))
print("Macro F1 tuned: ", f1_score(val_targets, (val_probs >= best_thresholds).astype(int),
                                   average="macro", zero_division=0))